### **House Prices - Advanced Regression Techniques**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [6]:
test.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


In [13]:
print(train.shape)
print(test.shape)

(1460, 81)
(1459, 80)


In [7]:
# check missing values
missing_train = train.isnull().sum().sort_values(ascending=False)
missing_test = test.isnull().sum().sort_values(ascending=False)

print("Missing values in Train dataset:")
print(missing_train[missing_train > 0])

print("\nMissing values in Test dataset:")
print(missing_test[missing_test > 0])

Missing values in Train dataset:
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

Missing values in Test dataset:
PoolQC          1456
MiscFeature     1408
Alley           1352
Fence           1169
MasVnrType       894
FireplaceQu      730
LotFrontage      227
GarageYrBlt       78
GarageCond        78
GarageFinish      78
GarageQual        78
GarageType        76
BsmtCond          45
BsmtQual          44
BsmtExposure      44
BsmtFinType1      42
BsmtFinType2      42
MasVnrArea        15
MSZoning           4
BsmtHalfBath       2
Utilities          2
Functional         2
BsmtFullBath       2
BsmtFinSF1         1
Exterior1st       

In [14]:
# Identify numerical and categorical columns
numerical_cols = train.select_dtypes(include=[np.number]).columns
categorical_cols = train.select_dtypes(exclude=[np.number]).columns

In [18]:
# ------------------------
# 1. Drop columns with too many missing values
# ------------------------
missing_threshold = 0.5  # 50%
drop_cols = [col for col in train.columns if train[col].isnull().mean() > missing_threshold]

train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

# Update categorical_cols after dropping columns
numerical_cols = train.select_dtypes(include=[np.number]).columns
categorical_cols = train.select_dtypes(exclude=[np.number]).columns


# ------------------------
# 2. Fill missing values (numerical)
# ------------------------
for col in numerical_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    if col in test.columns: # Check if column exists in test set
        test[col] = test[col].fillna(median_val)


# ------------------------
# 3. Fill missing values (categorical)
# ------------------------
for col in categorical_cols:
    train[col] = train[col].fillna("None")
    if col in test.columns: # Check if column exists in test set
        test[col] = test[col].fillna("None")


# ------------------------
# 4. Special case handling
# ------------------------
# LotFrontage -> fill with median per neighborhood
if "LotFrontage" in train.columns:
    train["LotFrontage"] = train.groupby("Neighborhood")["LotFrontage"].transform(
        lambda x: x.fillna(x.median())
    )
    if "LotFrontage" in test.columns: # Check if column exists in test set
        test["LotFrontage"] = test.groupby("Neighborhood")["LotFrontage"].transform(
            lambda x: x.fillna(x.median())
        )

# GarageYrBlt -> fill with 0 if missing
if "GarageYrBlt" in train.columns:
    train["GarageYrBlt"] = train["GarageYrBlt"].fillna(0)
    if "GarageYrBlt" in test.columns: # Check if column exists in test set
        test["GarageYrBlt"] = test["GarageYrBlt"].fillna(0)

# MasVnrArea -> fill with 0
if "MasVnrArea" in train.columns:
    train["MasVnrArea"] = train["MasVnrArea"].fillna(0)
    if "MasVnrArea" in test.columns: # Check if column exists in test set
        test["MasVnrArea"] = test["MasVnrArea"].fillna(0)

# MasVnrType -> fill with None
if "MasVnrType" in train.columns:
    train["MasVnrType"] = train["MasVnrType"].fillna("None")
    if "MasVnrType" in test.columns: # Check if column exists in test set
        test["MasVnrType"] = test["MasVnrType"].fillna("None")

print("✅ Data cleaning done!")
print("Train shape:", train.shape)
print("Test shape:", test.shape)

✅ Data cleaning done!
Train shape: (1460, 76)
Test shape: (1459, 75)


1. Encode categorical variables
XGBoost and LightGBM don’t accept text — they need numbers.
Use Label Encoding (simple, works well for tree models).

In [19]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical features
for col in train.select_dtypes(exclude=["number"]).columns:
    le = LabelEncoder()
    le.fit(list(train[col].astype(str)) + list(test[col].astype(str)))
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

2. Align train & test columns
(make sure both have the same features after cleaning)

In [20]:
train, test = train.align(test, join="left", axis=1, fill_value=0)

### XGBoost model training.

In [25]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# Separate features (X) and target (y)
X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]

# Train-test split (for validation)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Define XGBoost model
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Train the model
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

# Validation predictions
y_pred = xgb_model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
print("Validation RMSE:", rmse)

# Align columns of X and test before prediction
test_aligned = test.reindex(columns=X.columns, fill_value=0)

# Predict on test set
test_preds = xgb_model.predict(test_aligned)

# Save submission
submission = pd.DataFrame({"Id": test.index, "SalePrice": test_preds})
submission.to_csv("xgb_submission.csv", index=False)

print("✅ XGBoost training done, predictions saved to xgb_submission.csv")

Validation RMSE: 25419.338779755857
✅ XGBoost training done, predictions saved to xgb_submission.csv


In [26]:
submission = pd.DataFrame({
    "Id": test["Id"],   # use the actual Id column
    "SalePrice": test_preds
})
submission.to_csv("xgb_submission.csv", index=False)

In [27]:
from sklearn.metrics import mean_squared_error
import numpy as np

y_pred = xgb_model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
print("Validation RMSE:", rmse)

Validation RMSE: 25419.338779755857


In [28]:
# Average SalePrice in training set
avg_price = y_train.mean()

# RMSE as a percentage
rmse_percent = (rmse / avg_price) * 100
print("Validation RMSE (% of average price):", round(rmse_percent, 2), "%")

Validation RMSE (% of average price): 14.01 %


In [29]:
accuracy = 100 - (rmse / y_train.mean() * 100)
print("Approximate prediction accuracy:", round(accuracy, 2), "%")

Approximate prediction accuracy: 85.99 %
